## IIR Recursive filter

In [4]:
# Libraries
import cv2
import matplotlib.pyplot as plt
import numpy as np
import time
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

In [2]:
# Reading the frames from the video
vidcap = cv2.VideoCapture('D:/VR-Visuarl Recognition/v_Biking_g14_c04.avi')
success, image = vidcap.read()

# Input sequences
frames_arr = []
while success:
    success, image = vidcap.read()
    
    if success == False:
        break
        
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)    
    frames_arr.append(image)

### Temporal smoothing and differentiating

References: Miguel Velhote slides and https://www.cs.toronto.edu/~fleet/research/Papers/iir-flow.pdf

In [6]:
# Defining frames per second and size of the video
fps = vidcap.get(cv2.CAP_PROP_FPS)
frame_width = int(vidcap.get(3)) 
frame_height = int(vidcap.get(4)) 
size = (frame_width, frame_height)

# Defining the video files
result_smooth = cv2.VideoWriter('D:/VR-Visuarl Recognition/iir_smooth.mp4', cv2.VideoWriter_fourcc('m','p','4','v'), fps, size)
result_diff = cv2.VideoWriter('D:VR-Visuarl Recognition/iir_diff.mp4', cv2.VideoWriter_fourcc('m','p','4','v'), fps, size)

start_time = time.time()
tot_time = 0

smoothed_sequence = []
differenced_sequence = []

# Recursive IIR filter 
tau = 1
w_t1 = frames_arr[1]
w_t2 = frames_arr[0]
y_t1 = frames_arr[1]

for n in range(2, len(frames_arr)):
    
    q = tau / (tau + 2)
    r = (tau - 2) / (tau + 2)

    # First stage of the cascade
    w_t = frames_arr[n] - 2*r*w_t1 - (r**2)*w_t2
    R2_t = (q**2)*w_t + 2*(q**2)*w_t1 + (q**2)*w_t2

    # Second stage of the cascade
    y_t = R2_t - r*y_t1
    R3_t = q*y_t + q*y_t1

    # Diff image sequence
    diff = tau*(R2_t - R3_t)
    
    # Normalization
    img_smooth = R3_t*255.0 / np.max(R3_t)
    img_diff = diff*255.0 / np.max(diff)
    
    # Time computing
   
    # Writing
    smoothed_sequence.append(img_smooth.astype('uint8'))
    result_smooth.write(img_smooth.astype('uint8'))
    
    differenced_sequence.append(img_diff.astype('uint8'))
    result_diff.write(img_diff.astype('uint8'))
    
    w_t1 = w_t*1
    w_t2 = w_t1*1
    y_t1 = y_t*1
    
result_smooth.release()
result_diff.release()
tot_time = time.time() - start_time
print(f'Computation time per image: ', tot_time/len(frames_arr))

Computation time per image:  1.9339293119234917


In [9]:
frames_arr[0].shape, len(frames_arr), fps

((240, 320, 3), 156, 29.97002997002997)

### Metrics

In [7]:
def calculate_metrics(original, processed):
    psnr_value = peak_signal_noise_ratio(original, processed)
    win_size = min(original.shape[0], original.shape[1]) - 1  
    channel_axis = 2  
    ssim_value, _ = structural_similarity(original, processed, full=True, win_size=win_size, channel_axis=channel_axis)
 
    return psnr_value, ssim_value

def calculate_michelson_contrast(image):
    min_intensity = np.min(image)
    max_intensity = np.max(image)
    contrast = float((max_intensity - min_intensity) / (max_intensity + min_intensity)) * 100
    return contrast
 
def calculate_sharpness(image):
    # Convert the image to grayscale
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # Use Laplacian operator to calculate image gradient
    laplacian = cv2.Laplacian(gray_image, cv2.CV_64F)
    # Calculate sharpness as the variance of the Laplacian
    sharpness = np.var(laplacian)
    return sharpness

In [8]:
smooth_contrasts = []
smooth_sharpnesses = []
smooth_psnr = []
smooth_ssim = []
diff_contrasts = []
diff_sharpnesses = []
diff_psnr = []
diff_ssim = []

for smoothed_frame, differenced_frame, orig_frame in zip(smoothed_sequence, differenced_sequence, frames_arr):
    contrast_smoothed = calculate_michelson_contrast(smoothed_frame)
    sharpness_smoothed = calculate_sharpness(smoothed_frame)
    psnr_smoothed, ssim_smoothed  = calculate_metrics(orig_frame, smoothed_frame)
    contrast_differenced = calculate_michelson_contrast(differenced_frame)
    sharpness_differenced = calculate_sharpness(differenced_frame)
    psnr_differenced, ssim_differenced = calculate_metrics(orig_frame, differenced_frame)
    
    smooth_contrasts.append(contrast_smoothed)
    smooth_sharpnesses.append(sharpness_smoothed)
    smooth_psnr.append(psnr_smoothed)
    smooth_ssim.append(ssim_smoothed)
    diff_contrasts.append(contrast_differenced)
    diff_sharpnesses.append(sharpness_differenced)
    diff_psnr.append(psnr_differenced)
    diff_ssim.append(ssim_differenced)

print(f"Smoothed - Average Contrast: {np.mean(smooth_contrasts)}")
print(f"Smoothed - Average Sharpness: {np.mean(smooth_sharpnesses)}")
print(f"Smoothed - Average PSNR: {np.mean(smooth_psnr)}")
print(f"Smoothed - Average SSIM: {np.mean(smooth_ssim)}\n")

print(f"Differenced - Average Contrast: {np.mean(contrast_smoothed)}")
print(f"Differenced - Average Sharpness: {np.mean(diff_contrasts)}")
print(f"Differenced - Average PSNR: {np.mean(diff_psnr)}")
print(f"Differenced - Average SSIM: {np.mean(diff_ssim)}")

Smoothed - Average Contrast: 100.0
Smoothed - Average Sharpness: 96.79773611313792
Smoothed - Average PSNR: 29.958660033882882
Smoothed - Average SSIM: 0.989952385234212

Differenced - Average Contrast: 100.0
Differenced - Average Sharpness: 100.0
Differenced - Average PSNR: 7.925539002597143
Differenced - Average SSIM: 0.3205620583488553
